# 0.0. Setup

In [1]:
import os
import gc
import json
import copy
import requests
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

import networkx as nx
import geopandas as gpd
from shapely.geometry import Point
from difflib import SequenceMatcher

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from category_encoders import BinaryEncoder

from scipy.spatial import cKDTree
from scipy.sparse import lil_matrix
from scipy.sparse.csgraph import dijkstra
from pyproj import Transformer

from matplotlib.colors import Normalize
import matplotlib.cm as cm

# None means 'no limit'
pd.set_option("display.max_columns", None)

PROJECT_DIR = Path.cwd().parent
# if 'modules' not in os.listdir(Path.cwd()):
#     os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
DATA_PUBLIC_DIR = PROJECT_DIR / "data/public"
DATA_PRIVATE_DIR = PROJECT_DIR / "data/private"
DATA_PROCESSED_DIR = PROJECT_DIR / "data/processed"
OUTPUT_DIR = PROJECT_DIR / "output"

# 1.0. Load Data

## 1.1. School Information

In [2]:
# Load school information
fpath = str(OUTPUT_DIR / "processed_project_bukas_school_information.parquet")
sch_info = pd.read_parquet(fpath)
print(f"\nSchool info shape: {sch_info.shape}")

# Create slim version with just the columns we need
essential_cols = [
    "school_id",
    "school_name",
    "old_region",
    "division",
    "sector",
    "offers_es",
    "offers_jhs",
    "offers_shs",
]
sch_info_slim = sch_info[essential_cols].copy()
sch_info_slim["school_id"] = sch_info_slim["school_id"].astype(str)

print(f"School info slim shape: {sch_info_slim.shape}")
print(f"\nSample:")
display(sch_info_slim.head())


School info shape: (61442, 20)
School info slim shape: (61442, 8)

Sample:


,school_id,school_name,old_region,division,sector,offers_es,offers_jhs,offers_shs
0,100001,Apaleng-Libtong ES,Region I,Ilocos Norte,Public,True,False,False
1,100002,Bacarra CES,Region I,Ilocos Norte,Public,True,False,False
2,100003,Buyon ES,Region I,Ilocos Norte,Public,True,False,False
3,100004,Ganagan Elementary School,Region I,Ilocos Norte,Public,True,False,False
4,100005,Macupit ES,Region I,Ilocos Norte,Public,True,False,False


## 1.2. Distance Matrix

In [3]:
# Load distance matrix
fpath = str(OUTPUT_DIR / "processed_school_distance_matrix_osrm.npy")
distance_matrix = np.load(fpath)

fpath = str(OUTPUT_DIR / "processed_school_distance_matrix_index.json")
with open(fpath, "r") as f:
    index_data = json.load(f)
    school_ids = index_data["school_ids"]
    school_id_to_idx = index_data["school_id_to_indices"]
print(f"Distance matrix shape: {distance_matrix.shape}")

# Handle duplicate school IDs (keep first index, drop second)
# Some schools have 2 indices due to separate ES/JHS building coordinates
# dupe_ids = [(schid, ids) for schid, ids in school_id_to_idx.items() if isinstance(ids, list) and len(ids) == 2]
# idxs_to_drop = sorted([max(int(tup[1][0]), int(tup[1][1])) for tup in dupe_ids])
# print(f"Duplicate IDs to resolve: {len(dupe_ids)}")

# Trim matrix by removing duplicate rows/columns
dm = distance_matrix.copy()
# dm = np.delete(dm, idxs_to_drop, axis=1)
# dm = np.delete(dm, idxs_to_drop, axis=0)
# print(f"Trimmed distance matrix: {dm.shape}")

# # Create old_idx -> new_idx mapping (accounting for dropped indices)
# old_to_new_idx = {}
# new_idx = 0
# for old_idx in range(distance_matrix.shape[0]):
#     if old_idx not in idxs_to_drop:
#         old_to_new_idx[old_idx] = new_idx
#         new_idx += 1

# Create school_id -> new_idx mapping
re_schids_to_idx = copy.deepcopy(school_id_to_idx)
# for schid, idx_val in school_id_to_idx.items():
#     # Get the first index (could be list or single value)
#     if isinstance(idx_val, list):
#         old_idx = int(idx_val[0])  # Keep first index
#     else:
#         old_idx = int(idx_val)

#     # Map to new index if it wasn't dropped
#     if old_idx in old_to_new_idx:
#         re_schids_to_idx[schid] = old_to_new_idx[old_idx]

re_schids = list(re_schids_to_idx.keys())
print(f"Schools in distance matrix: {len(re_schids):,}")

del distance_matrix  # Free memory

Distance matrix shape: (13106, 13106)
Schools in distance matrix: 13,106


## 1.3. Student Flow

In [4]:
fpath = str(OUTPUT_DIR / "grade_7_student_flow_table_sy2324.parquet")
student_flow = pd.read_parquet(fpath)
print(f"Student flow shape: {student_flow.shape}")
print(f"Columns: {student_flow.columns.tolist()}")

Student flow shape: (288328, 4)
Columns: ['school_id_origin', 'school_id_destination', 'count_non_beneficiary', 'count_esc_beneficiary']


In [46]:
display(student_flow.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary
0,100000,300378,8.0,NaN
1,100000,300388,25.0,NaN
2,100000,305334,1.0,NaN
3,100000,500057,2.0,NaN
4,100001,300002,6.0,NaN


## 1.4. Public School Seats

In [5]:
fpath = str(OUTPUT_DIR / "processed_public_seats.parquet")
public_seats = pd.read_parquet(fpath)
print(f"Public seats shape: {public_seats.shape}")

Public seats shape: (54360, 3)


## 1.5. JHS Enrollment

In [6]:
fpath = str(OUTPUT_DIR / "processed_project_bukas_school_enrollment.parquet")
enrollment = pd.read_parquet(fpath)

mask = (enrollment["school_year"] == "2023_24") & (
    enrollment["education_level"] == "junior_high_school"
)
jhs_enrollment = (
    enrollment[mask].groupby(["school_id"], as_index=False)["enrollment_count"].sum()
)
dict_jhs_enrollment = {
    schid: enr
    for schid, enr in zip(
        jhs_enrollment["school_id"], jhs_enrollment["enrollment_count"]
    )
}
print(f"JHS enrollment records: {len(dict_jhs_enrollment):,}")

JHS enrollment records: 16,376


In [7]:
display(enrollment.head())

,school_id,enrollment_label,enrollment_count,grade_level,education_level,school_year
0,100001,enr_kindergarten,6.0,kindergarten,elementary,2022_23
1,100002,enr_kindergarten,56.0,kindergarten,elementary,2022_23
2,100003,enr_kindergarten,23.0,kindergarten,elementary,2022_23
3,100004,enr_kindergarten,10.0,kindergarten,elementary,2022_23
4,100005,enr_kindergarten,7.0,kindergarten,elementary,2022_23


In [8]:
display(jhs_enrollment.isna().sum())

school_id           0
enrollment_count    0
dtype: int64

## 1.6. School Coordinates

In [9]:
fpath = str(OUTPUT_DIR / "all_schools_coordinates_ncr_region4a_region3.parquet")
coordinates = pd.read_parquet(fpath)
print(f"Coordinates shape: {coordinates.shape}")

Coordinates shape: (13131, 4)


## 1.7. ESC Slots

In [10]:
fpath = str(OUTPUT_DIR / "processed_esc_slots.parquet")
esc_slots = pd.read_parquet(fpath)
print(f"ESC slots shape: {esc_slots.shape}")

ESC slots shape: (4638, 6)


In [11]:
display(esc_slots.head(3))

,school_id,esc_school_id,school_name,slots_total,slots_unutilized,has_deped_school_id
0,418013,303929,"One La Salle Educational Foundation, Inc.",38,6,True
1,408365,1104375,Newfoundland Arts and Science Academy of Tagum...,50,27,True
2,406245,1504390,4th Watch Maranatha Christian Academy of Bagui...,94,67,True


In [12]:
esc_slots[esc_slots["esc_school_id"].duplicated()].sort_values(
    by="esc_school_id", ascending=True
)

,school_id,esc_school_id,school_name,slots_total,slots_unutilized,has_deped_school_id


## 1.8. School Fees (GASTPE)

In [13]:
fpath = str(OUTPUT_DIR / "processed_gastpe_data.parquet")
gastpe = pd.read_parquet(fpath)
print(f"GASTPE data shape: {gastpe.shape}")

# Filter to ESC-delivering schools with JHS tuition data
esc_tuition = gastpe[
    (gastpe["is_esc_delivering"] == True) & (gastpe["average_jhs_total_fees"].notna())
][["school_id", "average_jhs_total_fees"]].copy()
esc_tuition.columns = ["school_id", "tuition_jhs"]
print(f"ESC schools with tuition data: {esc_tuition.shape[0]:,}")

GASTPE data shape: (5188, 5)
ESC schools with tuition data: 3,621


## 1.9. Private School Size (JHS Enrollment)

Calculate school "size" based on JHS enrollment to prioritize larger private schools in redistribution.

**Percentile Rank Approach:**
- Calculate `size_bonus` as the percentile rank of each school's JHS enrollment
- Range: 0.0 (smallest school) to 1.0 (largest school)
- Continuous score — no arbitrary binning
- Larger schools receive proportionally higher bonus in optimization

In [14]:
enrollment.head(1)

,school_id,enrollment_label,enrollment_count,grade_level,education_level,school_year
0,100001,enr_kindergarten,6.0,kindergarten,elementary,2022_23


In [15]:
# Filter to JHS enrollment for 2023_24 (all sectors)
mask_jhs = (enrollment["school_year"] == "2023_24") & (
    enrollment["education_level"] == "junior_high_school"
)
jhs_enr_all = (
    enrollment[mask_jhs].groupby("school_id", as_index=False)["enrollment_count"].sum()
)

# Join with school info to get sector
jhs_enr_all = jhs_enr_all.merge(
    sch_info_slim[["school_id", "sector"]], on="school_id", how="left"
)

# Filter to private schools only
private_jhs_enrollment = jhs_enr_all[jhs_enr_all["sector"] == "Private"].copy()
private_jhs_enrollment = private_jhs_enrollment[["school_id", "enrollment_count"]]
private_jhs_enrollment.columns = ["school_id", "private_jhs_enrollment"]
print(f"Private schools with JHS enrollment: {private_jhs_enrollment.shape[0]:,}")

# Calculate size_bonus using percentile rank (0.0 to 1.0)
# This gives each school a unique score based on where it falls in the distribution
private_jhs_enrollment["size_bonus"] = private_jhs_enrollment[
    "private_jhs_enrollment"
].rank(pct=True)

# Summary statistics
print(f"\nEnrollment distribution:")
print(f"  Min: {private_jhs_enrollment['private_jhs_enrollment'].min():,.0f} students")
print(
    f"  25th percentile: {private_jhs_enrollment['private_jhs_enrollment'].quantile(0.25):,.0f} students"
)
print(
    f"  Median: {private_jhs_enrollment['private_jhs_enrollment'].median():,.0f} students"
)
print(
    f"  75th percentile: {private_jhs_enrollment['private_jhs_enrollment'].quantile(0.75):,.0f} students"
)
print(f"  Max: {private_jhs_enrollment['private_jhs_enrollment'].max():,.0f} students")

print(f"\nSize bonus (percentile rank):")
print(
    f"  Range: [{private_jhs_enrollment['size_bonus'].min():.3f}, {private_jhs_enrollment['size_bonus'].max():.3f}]"
)
print(f"  Median school gets size_bonus ≈ 0.50")

# Show example mapping
print(f"\nExample size_bonus values:")
for pct in [0.1, 0.25, 0.5, 0.75, 0.9]:
    enr_at_pct = private_jhs_enrollment["private_jhs_enrollment"].quantile(pct)
    print(
        f"  {pct*100:.0f}th percentile ({enr_at_pct:,.0f} students) → size_bonus ≈ {pct:.2f}"
    )

Private schools with JHS enrollment: 5,620

Enrollment distribution:
  Min: 0 students
  25th percentile: 39 students
  Median: 110 students
  75th percentile: 252 students
  Max: 2,490 students

Size bonus (percentile rank):
  Range: [0.018, 1.000]
  Median school gets size_bonus ≈ 0.50

Example size_bonus values:
  10th percentile (12 students) → size_bonus ≈ 0.10
  25th percentile (39 students) → size_bonus ≈ 0.25
  50th percentile (110 students) → size_bonus ≈ 0.50
  75th percentile (252 students) → size_bonus ≈ 0.75
  90th percentile (477 students) → size_bonus ≈ 0.90


## 1.10. ESC Certification Ratings

In [16]:
fpath = str(OUTPUT_DIR / "processed_esc_certification_rating.parquet")
esc_ratings = pd.read_parquet(fpath)
print(esc_ratings.shape)

(899, 6)


In [17]:
display(esc_ratings.head(3))

,school_id,esc_school_id,annex,annex_label,has_deped_school_id,rating_rank
0,None,104519,D,JHS Visited for Certification,False,2
1,None,104558,D,JHS Visited for Certification,False,4
2,None,104552,D,JHS Visited for Certification,False,4


# 2.0. Build Congestion Table

## 2.1. Filter Schools to NCR, Region IV-A, & Region III

In [18]:
# Filter to public JHS in NCR and Region IV-A
mask = (
    (sch_info_slim["sector"] == "Public")
    & (sch_info_slim["offers_jhs"] == True)
    & (sch_info_slim["old_region"].isin(["NCR", "Region IV-A", "Region III"]))
)
sch_public_jhs = sch_info_slim.loc[mask].copy()
drop_cols = [col for col in sch_public_jhs.columns if re.search(r"offers_|sector", col)]
sch_public_jhs.drop(columns=drop_cols, inplace=True)
print(f"Public JHS in NCR/Region IV-A: {sch_public_jhs.shape[0]:,}")

Public JHS in NCR/Region IV-A: 2,032


## 2.2. Join Enrollment Data

In [19]:
# Create enrollment lookup
dict_jhs_enrollment = jhs_enrollment.set_index("school_id")[
    "enrollment_count"
].to_dict()

# Join enrollment to school info
tmp_jn = sch_public_jhs.copy()
tmp_jn["enrollment_jhs"] = tmp_jn["school_id"].map(dict_jhs_enrollment)

# Check coverage
missing_enrollment = tmp_jn["enrollment_jhs"].isna().sum()
print(f"Schools with enrollment data: {tmp_jn.shape[0] - missing_enrollment:,}")
print(f"Schools missing enrollment: {missing_enrollment:,}")

# Drop rows with missing enrollment (Feb. 4, 2026)
mask = tmp_jn["enrollment_jhs"].isna()
tmp_jn = tmp_jn.loc[~mask]
print(f"\nSchools with enrollment data without missing: {tmp_jn.shape}")

Schools with enrollment data: 1,986
Schools missing enrollment: 46

Schools with enrollment data without missing: (1986, 5)


In [20]:
# display(tmp_jn)

### Address NaNs
(Feb. 4, 2026) I decided not to address schools whose enrollment are NaN.

In [21]:
# # Load raw BUKAS Excel SY 23-24
# fpath = str(DATA_PROCESSED_DIR / "project_bukas_enrollment_2023-24.csv")
# csv = pd.read_csv(fpath)
# print(csv.shape)

# jhs_columns = (
#     ['school_id','school_name','region','division']
#     + ['offers_es', 'offers_jhs', 'offers_shs']
#     + [col for col in csv.columns if re.search(r'g7|g8|g9|g10', col)]
# )
# csv_jhs = csv[jhs_columns].copy()
# display(csv_jhs.head(1))

In [22]:
# enr_nans = tmp_jn[tmp_jn['enrollment_jhs'].isna()].copy()
# print(enr_nans.shape)

In [23]:
# display(enrollment.head(1))

In [24]:
# n = 13
# display(enr_nans.iloc[n:,:].head())

In [25]:
# schids = ["502882", "502830"]
# sch_name = r"sta.lucia|san jose\b"

In [26]:
# mask = (
#     (sch_info['school_id'].isin(schids))
# )
# # Processed school information from 3 SYs
# display(sch_info.loc[mask])

# # Raw BUKAS enrollment .CSV SY 23-24
# mask = (
#     (csv_jhs['school_id'].isin(schids))
#     | (csv_jhs['school_name'].str.contains(sch_name, flags=re.IGNORECASE))
#     & (csv_jhs['region'].isin(['Region IV-A','Region III','NCR']))
# )
# display(csv_jhs.loc[mask].sort_values(by=['school_name'], ascending=False))

In [27]:
# mask = (jhs_enrollment['school_id'].isin(schids))
# jhs_enrollment.loc[mask]

In [28]:
# # Manually encoded school ID pairs
# new_schids = {
#     "301291":"502667", # check
#     "301525":"502666", # check
#     "301525":"502670",
#     "306391":"306391", # School established in SY 24-25
#     "502817":"502817", # School enrollment starts in SY 24-25
#     "502854":"502854", # School enrollment starts in SY 24-25
#     "502790":"502790", # School enrollment starts in SY 24-25
#     "502832":"502832", # School enrollment starts in SY 24-25
#     "502805":"502805", # School enrollment starts in SY 24-25
#     "502881":"502881", # School enrollment starts in SY 24-25
#     "502838":"502838", # School enrollment starts in SY 24-25
#     # checkpoint: 502806 is Patola Integrated School
#     "502806":"502806", # School enrollment starts in SY 24-25
#     "502807":"502806", # School enrollment starts in SY 24-25
# }

## 2.3. Join Seat Data

In [29]:
# Filter seats to JHS level
jhs_seats = public_seats[public_seats["education_level"] == "Junior High School"][
    ["school_id", "seat_count"]
].copy()

# Join seats
enr_seats = tmp_jn.merge(jhs_seats, on="school_id", how="left")

# Check coverage
missing_seats = enr_seats["seat_count"].isna().sum()
print(f"Schools with seat data: {enr_seats.shape[0] - missing_seats:,}")
print(f"Schools missing seats: {missing_seats:,}")

Schools with seat data: 1,870
Schools missing seats: 116


## 2.4. Impute Missing Seats (KNN)

For schools missing seat data, we impute using K-Nearest Neighbors based on enrollment within the same region/division.

In [30]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

df_imp = enr_seats.copy()

# Identify rows needing imputation
mask = df_imp["seat_count"].isna()
print(f"Rows to impute: {mask.sum():,}")

# Prepare features for KNN
cat_cols = ["old_region", "division"]
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_imp[f"{col}_enc"] = le.fit_transform(df_imp[col].astype(str))
    le_dict[col] = le

# KNN imputation
feature_cols = ["old_region_enc", "division_enc", "enrollment_jhs"]
imputer = KNNImputer(n_neighbors=5)

# Fit on rows with seat data, transform all
df_for_impute = df_imp[feature_cols + ["seat_count"]].copy()
df_imputed_arr = imputer.fit_transform(df_for_impute)
df_imp["seat_count"] = df_imputed_arr[:, -1]

# Track imputed rows
df_imp["is_seats_imputed"] = mask

# Clean up
df_imp = df_imp.drop(columns=[f"{col}_enc" for col in cat_cols])
print(f"Imputation complete. Imputed: {df_imp['is_seats_imputed'].sum():,}")

Rows to impute: 116
Imputation complete. Imputed: 116


## 2.5. Calculate Congestion Metrics

**Key metrics:**
- `seat_utilization_jhs`: Enrollment / Seats (>1 means overcrowded)
- `count_aisle_learner_jhs`: Students without seats (enrollment - seats, min 0)

In [31]:
df_cong = df_imp.copy()

# Calculate congestion metrics
df_cong["seat_utilization_jhs"] = df_cong["enrollment_jhs"] / df_cong["seat_count"]
df_cong["count_aisle_learner_jhs"] = (
    df_cong["enrollment_jhs"] - df_cong["seat_count"]
).clip(lower=0)

# Summary statistics
total_schools = df_cong.shape[0]
congested = (df_cong["count_aisle_learner_jhs"] > 0).sum()
total_aisle = df_cong["count_aisle_learner_jhs"].sum()

print(f"Total public JHS analyzed: {total_schools:,}")
print(
    f"Congested schools (aisle learners > 0): {congested:,} ({congested/total_schools*100:.1f}%)"
)
print(f"Total aisle learners: {total_aisle:,.0f}")

Total public JHS analyzed: 1,986
Congested schools (aisle learners > 0): 1,267 (63.8%)
Total aisle learners: 768,463


In [32]:
display(df_cong.head(10))

,school_id,school_name,old_region,division,enrollment_jhs,seat_count,is_seats_imputed,seat_utilization_jhs,count_aisle_learner_jhs
0,300800,San Jose City National High School (formerly C...,Region III,San Jose City,4307.0,2431.0,False,1.771699,1876.0
1,300833,Porais National High School,Region III,San Jose City,678.0,409.0,False,1.657702,269.0
2,300868,Tondod National High School,Region III,San Jose City,510.0,425.0,False,1.200000,85.0
3,306076,Tayabo High School,Region III,San Jose City,422.0,359.0,False,1.175487,63.0
4,306077,Kita-Kita High School,Region III,San Jose City,634.0,620.0,False,1.022581,14.0
5,306805,Sto. Nino 3rd National High School,Region III,San Jose City,771.0,680.0,False,1.133824,91.0
6,500632,Bagong Sikat Integrated School,Region III,San Jose City,156.0,198.0,False,0.787879,0.0
7,501238,San Agustin Integrated School,Region III,San Jose City,285.0,132.0,False,2.159091,153.0
8,306817,Caanawan National High School,Region III,San Jose City,1175.0,1034.0,False,1.136364,141.0
9,300672,Aurora National Science HS,Region III,Aurora,403.0,406.0,False,0.992611,0.0


In [33]:
# Make sure that there are no NaNs by this section of the notebook
display(df_cong.isna().sum())

school_id                  0
school_name                0
old_region                 0
division                   0
enrollment_jhs             0
seat_count                 0
is_seats_imputed           0
seat_utilization_jhs       0
count_aisle_learner_jhs    0
dtype: int64

# 3.0. Prepare Optimization Data

## 3.1. Filter ESC Schools to Location

In [34]:
# Filter ESC schools to NCR/Region IV-A (schools in distance matrix)
mask = esc_slots["school_id"].isin(re_schids)
esc_in_loc = (
    esc_slots[["school_id", "slots_total", "slots_unutilized"]].loc[mask].copy()
)
print(f"ESC schools in NCR/Region IV-A/Region III: {esc_in_loc.shape[0]:,}")

ESC schools in NCR/Region IV-A/Region III: 1,447


In [35]:
display(esc_in_loc.head(1))

,school_id,slots_total,slots_unutilized
0,418013,38,6


## 3.2. Filter Student Flow to Location

In [36]:
# Filter flow to schools in NCR/Region IV-A
mask = (student_flow["school_id_origin"].isin(re_schids)) & (
    student_flow["school_id_destination"].isin(re_schids)
)
flow = student_flow[mask].copy()
print(f"Flow pairs in location: {flow.shape[0]:,}")

Flow pairs in location: 84,131


## 3.3. Build Distance Lookup (Origin ES to Private ESC)

For each origin elementary school, find all private ESC schools within the distance threshold.

In [37]:
# Get private ESC school IDs in distance matrix
private_esc_ids = esc_in_loc["school_id"].tolist()
private_in_dm = [sid for sid in private_esc_ids if sid in re_schids_to_idx]

# Get origin school IDs from flow data
public_jhs_ids = df_cong["school_id"].tolist()
origin_ids = (
    flow[flow["school_id_destination"].isin(public_jhs_ids)]["school_id_origin"]
    .unique()
    .tolist()
)
origins_in_dm = [sid for sid in origin_ids if sid in re_schids_to_idx]

print(f"Private ESC schools in distance matrix: {len(private_in_dm):,}")
print(f"Origin schools in distance matrix: {len(origins_in_dm):,}")

# Build origin-to-private distance lookup
MAX_DISTANCE_KM = 5  # Only consider private schools within 5km of origin

rows = []
for origin_id in origins_in_dm:
    origin_idx = re_schids_to_idx[origin_id]
    for private_id in private_in_dm:
        private_idx = re_schids_to_idx[private_id]
        distance_m = dm[origin_idx, private_idx]
        distance_km = distance_m / 1000

        if distance_km <= MAX_DISTANCE_KM and distance_m < np.inf:
            rows.append(
                {
                    "school_id_origin": origin_id,
                    "private_esc_id": private_id,
                    "distance_to_private_m": distance_m,
                    "distance_to_private_km": distance_km,
                }
            )

origin_to_private = pd.DataFrame(rows)
origin_to_private = origin_to_private.merge(
    esc_in_loc[["school_id", "slots_total", "slots_unutilized"]],
    left_on="private_esc_id",
    right_on="school_id",
    how="left",
).drop(columns="school_id")

print(
    f"Origin-private pairs within {MAX_DISTANCE_KM}km: {origin_to_private.shape[0]:,}"
)

Private ESC schools in distance matrix: 1,447
Origin schools in distance matrix: 8,613
Origin-private pairs within 5km: 69,710


In [38]:
display(origin_to_private.head())

,school_id_origin,private_esc_id,distance_to_private_m,distance_to_private_km,slots_total,slots_unutilized
0,101701,402298,1228.1,1.2281,151,0
1,101701,424800,2840.8,2.8408,20,20
2,102178,402570,4996.1,4.9961,50,32
3,102178,424703,4437.3,4.4373,45,12
4,102259,401645,2744.4,2.7444,6,0


## 3.4. Link Flow to Congested Public JHS

In [39]:
# Join flow data with congestion info
flow_to_congested = flow.merge(
    df_cong[
        [
            "school_id",
            "count_aisle_learner_jhs",
            "seat_utilization_jhs",
            "enrollment_jhs",
            "seat_count",
        ]
    ],
    left_on="school_id_destination",
    right_on="school_id",
    how="inner",
).drop(columns="school_id")

# Filter to only congested destinations
mask = flow_to_congested["count_aisle_learner_jhs"] > 0
flow_to_congested = flow_to_congested[mask].copy()

print(f"Flows to congested public JHS: {flow_to_congested.shape[0]:,}")
print(
    f"Total non-beneficiaries in these flows: {flow_to_congested['count_non_beneficiary'].sum():,.0f}"
)

Flows to congested public JHS: 36,164
Total non-beneficiaries in these flows: 400,936


## 3.5. Build Redirection Options

In [60]:
# Create redirection options: (origin, destination, private_esc) combinations
redirection_options = flow_to_congested.merge(
    origin_to_private, on="school_id_origin", how="inner"
)
print(f"Redirection options: {redirection_options.shape[0]:,}")

Redirection options: 433,515


In [61]:
display(origin_to_private.head())

,school_id_origin,private_esc_id,distance_to_private_m,distance_to_private_km,slots_total,slots_unutilized
0,101701,402298,1228.1,1.2281,151,0
1,101701,424800,2840.8,2.8408,20,20
2,102178,402570,4996.1,4.9961,50,32
3,102178,424703,4437.3,4.4373,45,12
4,102259,401645,2744.4,2.7444,6,0


## 3.6. Available ESC Slots

Use `slots_unutilized` directly from official stakeholder data (GASS ESC Slots dataset).

In [62]:
# Use official slots_unutilized data directly (no recalculation needed)
# Rename for consistency with downstream code
redirection_options = redirection_options.rename(
    columns={"slots_unutilized": "available_slots"}
)

# Drop slots_total if present (no longer needed)
redirection_options = redirection_options.drop(columns=["slots_total"], errors="ignore")

# Filter to options with available slots
redirection_options = redirection_options[
    redirection_options["available_slots"] > 0
].copy()

# Build esc_available for downstream compatibility (greedy, LP, exports)
esc_available = esc_in_loc.rename(
    columns={"slots_unutilized": "available_slots"}
).copy()

print(
    f"ESC schools with available slots > 0: {(esc_available['available_slots'] > 0).sum():,}"
)
print(f"Total available slots: {esc_available['available_slots'].sum():,.0f}")
print(f"Redirection options with available slots: {redirection_options.shape[0]:,}")

ESC schools with available slots > 0: 1,093
Total available slots: 24,745
Redirection options with available slots: 303,358


In [63]:
print(redirection_options.shape)
display(redirection_options.head())

(303358, 12)


,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count,private_esc_id,distance_to_private_m,distance_to_private_km,available_slots
1,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0,424800,2840.8,2.8408,20
3,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0,424800,2840.8,2.8408,20
5,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0,424800,2840.8,2.8408,20
6,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,402570,4996.1,4.9961,32
7,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,424703,4437.3,4.4373,12


### Inspect

In [64]:
display(student_flow.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary
0,100000,300378,8.0,NaN
1,100000,300388,25.0,NaN
2,100000,305334,1.0,NaN
3,100000,500057,2.0,NaN
4,100001,300002,6.0,NaN


In [65]:
# Get existing origin-destination pairs from actual student flow
existing_pairs = student_flow[
    ["school_id_origin", "school_id_destination"]
].drop_duplicates()
existing_pairs["has_existing_flow"] = True

# Merge to check if origin-private pair exists in actual flow
redirection_options = redirection_options.merge(
    existing_pairs,
    left_on=["school_id_origin", "private_esc_id"],
    right_on=["school_id_origin", "school_id_destination"],
    how="left",
    suffixes=("", "_drop"),
).drop(columns=["school_id_destination_drop"])
# display(redirection_options.head())

# Label: True if NEW (no existing flow), False if pair already exists
redirection_options["is_new_pair"] = redirection_options["has_existing_flow"].isna()
redirection_options = redirection_options.drop(columns=["has_existing_flow"])

# Summary
new_pairs = redirection_options["is_new_pair"].sum()
existing_pairs_count = (~redirection_options["is_new_pair"]).sum()
print(
    f"Redirection options with existing student flow: {existing_pairs_count:,} ({existing_pairs_count/len(redirection_options)*100:.1f}%)"
)
print(
    f"Redirection options requiring new flow: {new_pairs:,} ({new_pairs/len(redirection_options)*100:.1f}%)"
)

Redirection options with existing student flow: 64,102 (21.1%)
Redirection options requiring new flow: 239,256 (78.9%)


In [66]:
# How many students could be redirected via existing vs new flows?
by_pair_type = (
    redirection_options.groupby("is_new_pair")
    .agg(
        {
            "count_non_beneficiary": "sum",
            "available_slots": "sum",
            "private_esc_id": "nunique",
        }
    )
    .rename(index={True: "New Pairs", False: "Existing Pairs"})
)
print(by_pair_type)

                count_non_beneficiary  available_slots  private_esc_id
is_new_pair                                                           
Existing Pairs               786821.0          2011857            1044
New Pairs                   2713367.0          5261508            1071


In [67]:
display(redirection_options.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count,private_esc_id,distance_to_private_m,distance_to_private_km,available_slots,is_new_pair
0,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0,424800,2840.8,2.8408,20,True
1,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0,424800,2840.8,2.8408,20,True
2,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0,424800,2840.8,2.8408,20,True
3,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,402570,4996.1,4.9961,32,True
4,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,424703,4437.3,4.4373,12,True


## 3.7. Add Tuition and School Size Data

Add tuition costs and school size features to prioritize:
- **Cheaper schools** (lower tuition)
- **Larger schools** (9th-10th decile by JHS enrollment)

In [29]:
# Join tuition data
redirection_options = redirection_options.merge(
    esc_tuition, left_on="private_esc_id", right_on="school_id", how="left"
).drop(columns="school_id", errors="ignore")

# Impute missing tuition with median
median_tuition = redirection_options["tuition_jhs"].median()
redirection_options["tuition_jhs"] = redirection_options["tuition_jhs"].fillna(
    median_tuition
)
print(f"Median JHS tuition: ₱{median_tuition:,.0f}")

# Join school size data (size_bonus from percentile rank)
redirection_options = redirection_options.merge(
    private_jhs_enrollment[["school_id", "private_jhs_enrollment", "size_bonus"]],
    left_on="private_esc_id",
    right_on="school_id",
    how="left",
).drop(columns="school_id", errors="ignore")

# Handle missing enrollment (schools not in enrollment data)
redirection_options["private_jhs_enrollment"] = redirection_options[
    "private_jhs_enrollment"
].fillna(0)
redirection_options["size_bonus"] = redirection_options["size_bonus"].fillna(
    0.0
)  # No bonus for missing

# Summary of size_bonus distribution
print(f"\nSize bonus distribution in redirection options:")
print(f"  Schools with size data: {redirection_options['size_bonus'].gt(0).sum():,}")
print(f"  Schools missing size data: {redirection_options['size_bonus'].eq(0).sum():,}")
print(f"  Mean size_bonus: {redirection_options['size_bonus'].mean():.3f}")
print(f"  Median size_bonus: {redirection_options['size_bonus'].median():.3f}")

# Priority scoring parameters
TUITION_WEIGHT = 0.5  # Weight for tuition in priority score
SIZE_WEIGHT = 0.3  # Weight for size bonus (multiplied by size_bonus 0-1)

# Calculate normalized values per destination group
for dest_id in redirection_options["school_id_destination"].unique():
    mask = redirection_options["school_id_destination"] == dest_id
    options = redirection_options.loc[mask]

    if len(options) > 1:
        dist_min = options["distance_to_private_km"].min()
        dist_max = options["distance_to_private_km"].max()
        tuition_min = options["tuition_jhs"].min()
        tuition_max = options["tuition_jhs"].max()

        redirection_options.loc[mask, "norm_distance"] = (
            options["distance_to_private_km"] - dist_min
        ) / (dist_max - dist_min + 1e-6)
        redirection_options.loc[mask, "norm_tuition"] = (
            options["tuition_jhs"] - tuition_min
        ) / (tuition_max - tuition_min + 1e-6)
    else:
        redirection_options.loc[mask, "norm_distance"] = 0
        redirection_options.loc[mask, "norm_tuition"] = 0

# Calculate priority score (lower = better)
# Base: distance + tuition penalty
# Bonus: subtract SIZE_WEIGHT * size_bonus (larger schools get bigger subtraction)
redirection_options["priority_score"] = (
    redirection_options["norm_distance"]
    + TUITION_WEIGHT * redirection_options["norm_tuition"]
    - SIZE_WEIGHT * redirection_options["size_bonus"]
)

print(f"\nPriority score calculated for {redirection_options.shape[0]:,} options")
print(
    f"Priority score range: [{redirection_options['priority_score'].min():.3f}, {redirection_options['priority_score'].max():.3f}]"
)
print(
    f"  Formula: norm_distance + {TUITION_WEIGHT}×norm_tuition - {SIZE_WEIGHT}×size_bonus"
)
print(f"  size_bonus = percentile rank [0.0, 1.0]")

Median JHS tuition: ₱38,012

Size bonus distribution in redirection options:
  Schools with size data: 355,716
  Schools missing size data: 0
  Mean size_bonus: 0.641
  Median size_bonus: 0.662

Priority score calculated for 355,716 options
Priority score range: [-0.295, 1.436]
  Formula: norm_distance + 0.5×norm_tuition - 0.3×size_bonus
  size_bonus = percentile rank [0.0, 1.0]


In [27]:
display(esc_available.head(5))

,school_id,total_count_slots,current_beneficiaries,available_slots
0,401884,50,23.0,27.0
1,410053,50,22.0,28.0
2,425615,50,7.0,43.0
3,424351,50,34.0,16.0
4,406858,50,41.0,9.0


# 4.0. [ON HOLD] Greedy Optimization

**Algorithm:** Address the most congested public schools first. For each, select private partners that are nearest and most affordable (lowest priority_score).

## 4.1. Initialize Tracking Variables

In [30]:
display(flow_to_congested.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count
0,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0
2,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0
5,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0
6,101782,301008,6.0,NaN,127.0,1.481061,391.0,264.0
8,101788,301008,4.0,NaN,127.0,1.481061,391.0,264.0


In [31]:
# Initialize remaining capacity tracking
remaining_slots = esc_available.set_index("school_id")["available_slots"].to_dict()
remaining_aisle = df_cong.set_index("school_id")["count_aisle_learner_jhs"].to_dict()

# Track remaining non-beneficiaries per (origin, destination) flow
# This prevents over-redirecting from a single flow
remaining_non_benef = flow_to_congested.set_index(
    ["school_id_origin", "school_id_destination"]
)["count_non_beneficiary"].to_dict()

# Sort redirection options by: most congested destination first, then lowest priority score
redirection_options_sorted = redirection_options.sort_values(
    by=["count_aisle_learner_jhs", "priority_score"], ascending=[False, True]
)

print(f"Initialized tracking for {len(remaining_slots):,} private schools")
print(f"Initialized tracking for {len(remaining_aisle):,} congested public schools")
print(f"Initialized tracking for {len(remaining_non_benef):,} flows")

Initialized tracking for 1,402 private schools
Initialized tracking for 1,986 congested public schools
Initialized tracking for 36,164 flows


## 4.2. Run Greedy Algorithm

In [25]:
redirection_log = []

for _, row in redirection_options_sorted.iterrows():
    origin_id = row["school_id_origin"]
    pub_jhs_id = row["school_id_destination"]
    private_id = row["private_esc_id"]
    distance_km = row["distance_to_private_km"]
    tuition = row["tuition_jhs"]
    priority = row["priority_score"]

    # Check remaining capacity
    slots_left = remaining_slots.get(private_id, 0)
    aisle_left = remaining_aisle.get(pub_jhs_id, 0)

    # Get remaining non-beneficiaries for THIS specific flow
    flow_key = (origin_id, pub_jhs_id)
    non_benef_left = remaining_non_benef.get(flow_key, 0)

    if slots_left <= 0 or aisle_left <= 0 or non_benef_left <= 0:
        continue

    # Calculate students to redirect (use remaining non-beneficiaries, not original count)
    count_to_redirect = min(slots_left, aisle_left, non_benef_left)

    if count_to_redirect > 0:
        # Log the redirection
        redirection_log.append(
            {
                "school_id_origin": origin_id,
                "school_id_destination": pub_jhs_id,
                "private_esc_id": private_id,
                "count_redirected": count_to_redirect,
                "distance_to_private_km": distance_km,
                "tuition_jhs": tuition,
                "priority_score": priority,
            }
        )

        # Update tracking
        remaining_slots[private_id] -= count_to_redirect
        remaining_aisle[pub_jhs_id] -= count_to_redirect
        remaining_non_benef[flow_key] -= count_to_redirect

redirections = pd.DataFrame(redirection_log)
print(f"Total redirections logged: {len(redirections):,}")
print(f"Total students redirected: {redirections['count_redirected'].sum():,.0f}")

Total redirections logged: 5,070
Total students redirected: 25,807


## 4.3. Calculate Greedy Results

In [26]:
# Update congestion table with Greedy results
scenario1_congestion = df_cong.copy()
greedy_by_dest = (
    redirections.groupby("school_id_destination")["count_redirected"].sum().to_dict()
)

scenario1_congestion["redirected_greedy"] = (
    scenario1_congestion["school_id"].map(greedy_by_dest).fillna(0)
)
scenario1_congestion["aisle_learners_greedy"] = (
    scenario1_congestion["count_aisle_learner_jhs"]
    - scenario1_congestion["redirected_greedy"]
).clip(lower=0)

# Summary
baseline_aisle = df_cong["count_aisle_learner_jhs"].sum()
greedy_aisle = scenario1_congestion["aisle_learners_greedy"].sum()

print(f"\nGREEDY OPTIMIZATION RESULTS:")
print(f"  Baseline aisle learners: {baseline_aisle:,.0f}")
print(f"  After Greedy: {greedy_aisle:,.0f}")
print(
    f"  Reduction: {baseline_aisle - greedy_aisle:,.0f} ({((baseline_aisle - greedy_aisle)/baseline_aisle)*100:.1f}%)"
)
print(f"  Private ESC schools used: {redirections['private_esc_id'].nunique():,}")


GREEDY OPTIMIZATION RESULTS:
  Baseline aisle learners: 577,701
  After Greedy: 551,894
  Reduction: 25,807 (4.5%)
  Private ESC schools used: 947


# 5.0. [DEPRECATED] LP Optimization

**Objective:** Find the global optimal redistribution that maximizes total students redirected while respecting all constraints.

Uses PuLP linear programming library.

## 5.1. Setup LP Problem

In [29]:
# Install PuLP if needed
try:
    from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus, value
except ImportError:
    !pip install -qq pulp
    from pulp import LpMaximize, LpProblem, LpVariable, lpSum, LpStatus, value

# Prepare constraint dictionaries
flow_limits = (
    flow_to_congested.groupby(["school_id_origin", "school_id_destination"])[
        "count_non_beneficiary"
    ]
    .first()
    .to_dict()
)

slot_limits = esc_available.set_index("school_id")["available_slots"].to_dict()
aisle_limits = df_cong.set_index("school_id")["count_aisle_learner_jhs"].to_dict()

print(f"Flow constraints: {len(flow_limits):,}")
print(f"Slot constraints: {len(slot_limits):,}")
print(f"Aisle learner constraints: {len(aisle_limits):,}")

Flow constraints: 22,879
Slot constraints: 1,058
Aisle learner constraints: 1,098


## 5.2. Define Decision Variables

In [30]:
# Create LP problem
prob = LpProblem("Maximize_Student_Redistribution", LpMaximize)

# Create decision variables: x[i] = students redirected for option i
x = {}
for idx, row in redirection_options.iterrows():
    var_name = f"x_{idx}"
    max_val = min(
        row["count_non_beneficiary"],
        row["available_slots"],
        row["count_aisle_learner_jhs"],
    )
    x[idx] = LpVariable(var_name, lowBound=0, upBound=max_val, cat="Continuous")

print(f"Decision variables created: {len(x):,}")

Decision variables created: 284,670


## 5.3. Add Constraints

In [31]:
# Constraint 1: Flow limit (can't redirect more than non-beneficiaries per flow)
flow_groups = redirection_options.groupby(["school_id_origin", "school_id_destination"])
for (origin, dest), group in flow_groups:
    flow_key = (origin, dest)
    if flow_key in flow_limits:
        prob += (
            lpSum([x[idx] for idx in group.index]) <= flow_limits[flow_key],
            f"flow_{origin}_{dest}",
        )

# Constraint 2: Slot limit (can't exceed available slots per private school)
private_groups = redirection_options.groupby("private_esc_id")
for private_id, group in private_groups:
    if private_id in slot_limits:
        prob += (
            lpSum([x[idx] for idx in group.index]) <= slot_limits[private_id],
            f"slots_{private_id}",
        )

# Constraint 3: Aisle learner limit (can't redirect more than aisle learners per public school)
dest_groups = redirection_options.groupby("school_id_destination")
for dest_id, group in dest_groups:
    if dest_id in aisle_limits:
        prob += (
            lpSum([x[idx] for idx in group.index]) <= aisle_limits[dest_id],
            f"aisle_{dest_id}",
        )

print(f"Constraints added: {len(prob.constraints):,}")

Constraints added: 21,011


In [32]:
# Is the lowest decongestion 100% or seats == enrollment ba?

## 5.4. Define Objective Function

Maximize total students redirected, with:
- **Penalties** for distant and expensive schools
- **Bonus** for large schools (9th-10th decile)

In [46]:
# Based on DCM: penalty on cost is lower than distance

In [33]:
COST_PENALTY = 0.3  # Penalty for expensive schools
DISTANCE_PENALTY = 0.3  # Penalty for distant schools
SIZE_WEIGHT_LP = 0.3  # Weight for size bonus (multiplied by size_bonus 0-1)

# Objective: Maximize redirections with distance/cost penalties and size bonus
objective_terms = []
for idx, row in redirection_options.iterrows():
    # Base value: 1 per student redirected
    # Penalties: deductions for distance and cost
    # Bonus: addition proportional to size_bonus (0-1)
    coef = (
        1.0
        - (DISTANCE_PENALTY * row["norm_distance"])
        - (COST_PENALTY * row["norm_tuition"])
        + (SIZE_WEIGHT_LP * row["size_bonus"])
    )
    objective_terms.append(coef * x[idx])

prob += lpSum(objective_terms), "Total_Weighted_Redirections"
print(f"Objective function defined with {len(objective_terms):,} terms")
print(f"  - Distance penalty: {DISTANCE_PENALTY}")
print(f"  - Cost penalty: {COST_PENALTY}")
print(f"  - Size weight: {SIZE_WEIGHT_LP} × size_bonus")
print(
    f"    (Decile 10 gets +{SIZE_WEIGHT_LP:.2f}, Decile 5 gets +{SIZE_WEIGHT_LP * 0.44:.2f}, Decile 1 gets +0.00)"
)

Objective function defined with 284,670 terms
  - Distance penalty: 0.3
  - Cost penalty: 0.3
  - Size weight: 0.3 × size_bonus
    (Decile 10 gets +0.30, Decile 5 gets +0.13, Decile 1 gets +0.00)


In [36]:
print(redirection_options.shape)
display(redirection_options.head(10))

(284670, 18)


,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count,private_esc_id,distance_to_private_m,distance_to_private_km,available_slots,tuition_jhs,private_jhs_enrollment,size_bonus,norm_distance,norm_tuition,priority_score
0,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0,402298,1228.1,1.2281,19.0,59250.00,645.0,0.943594,0.245654,0.318053,0.121603
1,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0,424800,2840.8,2.8408,20.0,51840.00,7.0,0.070018,0.568239,0.271152,0.682810
2,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0,402298,1228.1,1.2281,19.0,59250.00,645.0,0.943594,0.245728,0.404860,0.165080
3,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0,424800,2840.8,2.8408,20.0,51840.00,7.0,0.070018,0.568410,0.345158,0.719984
4,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0,402298,1228.1,1.2281,19.0,59250.00,645.0,0.943594,0.245684,0.467463,0.196337
5,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0,424800,2840.8,2.8408,20.0,51840.00,7.0,0.070018,0.568308,0.398529,0.746567
6,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,402570,4996.1,4.9961,33.0,18029.15,40.0,0.257384,1.000000,0.055636,0.950602
7,102178,301277,36.0,NaN,34.0,1.045333,784.0,750.0,424703,4437.3,4.4373,12.0,14130.00,96.0,0.461121,0.878664,0.016401,0.748528
8,102259,301155,17.0,NaN,313.0,1.178043,2071.0,1758.0,401645,2744.4,2.7444,1.0,43600.00,32.0,0.213968,0.516341,0.663189,0.783745
9,102259,301155,17.0,NaN,313.0,1.178043,2071.0,1758.0,401646,2837.7,2.8377,49.0,22400.00,771.0,0.962989,0.536656,0.237785,0.366651


In [42]:
redirection_options[redirection_options["school_id_destination"].str.contains(r"^4")]

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count,private_esc_id,distance_to_private_m,distance_to_private_km,available_slots,tuition_jhs,private_jhs_enrollment,size_bonus,norm_distance,norm_tuition,priority_score


## 5.5. Solve LP

In [31]:
print("Solving LP...")
prob.solve()

print(f"Status: {LpStatus[prob.status]}")
print(f"Objective value: {value(prob.objective):,.2f}")

Solving LP...
Welcome to the CBC MILP Solver 
Version: 2.10.10 
Build Date: Sep 26 2023 

command line - /home/jupyter/.local/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/arm64/cbc /tmp/0243e1db4ca244c3b9459272090242b6-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/0243e1db4ca244c3b9459272090242b6-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 21016 COLUMNS
At line 1159697 RHS
At line 1180709 BOUNDS
At line 1465380 ENDATA
Problem MODEL has 21011 rows, 284670 columns and 854010 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 19106 (-1905) rows, 283406 (-1264) columns and 793278 (-60732) elements
0  Obj 228.05813 Dual inf 261937.23 (283405)
320  Obj 725611.11 Primal inf 1732366.9 (18565)
640  Obj 587831.47 Primal inf 1302211.1 (18116)
960  Obj 481813.28 Primal inf 1006156.5 (17481)
1280  Obj 418749.42 Primal inf 837303.88 (17015)
1600  Obj 367199.04 Prima

## 5.6. Extract LP Results

In [32]:
# Extract LP solution
redirection_options_lp = redirection_options.copy()
redirection_options_lp["count_redirected_lp"] = [
    value(x[idx]) for idx in redirection_options.index
]

# Filter to positive redirections
redirections_lp = redirection_options_lp[
    redirection_options_lp["count_redirected_lp"] > 0
].copy()

print(
    f"Total students redirected (LP): {redirections_lp['count_redirected_lp'].sum():,.0f}"
)

# Update congestion table with LP results
scenario1_lp_congestion = df_cong.copy()
redirections_by_dest_lp = (
    redirections_lp.groupby("school_id_destination")["count_redirected_lp"]
    .sum()
    .to_dict()
)

scenario1_lp_congestion["redirected_lp"] = (
    scenario1_lp_congestion["school_id"].map(redirections_by_dest_lp).fillna(0)
)
scenario1_lp_congestion["aisle_learners_lp"] = (
    scenario1_lp_congestion["count_aisle_learner_jhs"]
    - scenario1_lp_congestion["redirected_lp"]
).clip(lower=0)

# Summary
lp_aisle = scenario1_lp_congestion["aisle_learners_lp"].sum()

print(f"\nLP OPTIMIZATION RESULTS:")
print(f"  Baseline aisle learners: {baseline_aisle:,.0f}")
print(f"  After LP: {lp_aisle:,.0f}")
print(
    f"  Reduction: {baseline_aisle - lp_aisle:,.0f} ({((baseline_aisle - lp_aisle)/baseline_aisle)*100:.1f}%)"
)
print(f"  Private ESC schools used: {redirections_lp['private_esc_id'].nunique():,}")

Total students redirected (LP): 25,930

LP OPTIMIZATION RESULTS:
  Baseline aisle learners: 577,701
  After LP: 551,771
  Reduction: 25,930 (4.5%)
  Private ESC schools used: 947


# 6.0. Compare Results

In [33]:
print("=" * 70)
print("COMPARISON: GREEDY vs LP OPTIMIZATION")
print("=" * 70)
print(f"{'Metric':<35} {'Greedy':>15} {'LP':>15}")
print("-" * 70)
print(
    f"{'Students redirected':<35} {redirections['count_redirected'].sum():>15,.0f} {redirections_lp['count_redirected_lp'].sum():>15,.0f}"
)
print(f"{'Aisle learners remaining':<35} {greedy_aisle:>15,.0f} {lp_aisle:>15,.0f}")
print(
    f"{'Reduction from baseline':<35} {(baseline_aisle-greedy_aisle)/baseline_aisle*100:>14.3f}% {(baseline_aisle-lp_aisle)/baseline_aisle*100:>14.3f}%"
)
print(
    f"{'Private ESC schools used':<35} {redirections['private_esc_id'].nunique():>15,} {redirections_lp['private_esc_id'].nunique():>15,}"
)

# Average distance comparison
greedy_avg_dist = np.average(
    redirections["distance_to_private_km"], weights=redirections["count_redirected"]
)
lp_avg_dist = np.average(
    redirections_lp["distance_to_private_km"],
    weights=redirections_lp["count_redirected_lp"],
)
print(
    f"{'Avg redirect distance (km)':<35} {greedy_avg_dist:>15.2f} {lp_avg_dist:>15.2f}"
)

# Average tuition comparison
greedy_avg_tuition = np.average(
    redirections["tuition_jhs"], weights=redirections["count_redirected"]
)
lp_avg_tuition = np.average(
    redirections_lp["tuition_jhs"], weights=redirections_lp["count_redirected_lp"]
)
print(
    f"{'Avg tuition (PHP)':<35} {greedy_avg_tuition:>15,.0f} {lp_avg_tuition:>15,.0f}"
)
print("=" * 70)

COMPARISON: GREEDY vs LP OPTIMIZATION
Metric                                       Greedy              LP
----------------------------------------------------------------------
Students redirected                          25,807          25,930
Aisle learners remaining                    551,894         551,771
Reduction from baseline                      4.467%          4.488%
Private ESC schools used                        947             947
Avg redirect distance (km)                     2.51            1.03
Avg tuition (PHP)                            38,633          38,795


# 6.1. Blocked Demand Analysis

Analyze the **remaining demand** after LP optimization to identify:
1. Which private ESC schools are **capacity-constrained** (≥95% slot utilization)
2. How much **additional demand** could be served if these schools received more slots
3. Which schools should be prioritized for **slot expansion**

**Key Metric:** `unique_blocked_demand` = students exclusively assigned to each school (no double-counting)

In [ ]:
# Step 1: Calculate slots used by LP for each private ESC
slots_used_by_school = redirections_lp.groupby("private_esc_id")[
    "count_redirected_lp"
].sum()

# Step 2: Build capacity info for private ESC schools
capacity_info = esc_available[["school_id", "slots_total", "available_slots"]].copy()
capacity_info.columns = ["private_esc_id", "total_slots", "available_slots_before"]
capacity_info["slots_used_lp"] = (
    capacity_info["private_esc_id"].map(slots_used_by_school).fillna(0)
)
capacity_info["utilization_rate"] = (
    capacity_info["slots_used_lp"] / capacity_info["available_slots_before"]
)
capacity_info["utilization_rate"] = capacity_info["utilization_rate"].replace(
    [float("inf")], 0
)

# Step 3: Identify capacity-constrained schools (>= 95% utilized)
constrained_schools = set(
    capacity_info[capacity_info["utilization_rate"] >= 0.95]["private_esc_id"].tolist()
)
print(
    f"Capacity-constrained private ESC schools (>=95% utilized): {len(constrained_schools):,}"
)

In [35]:
# Step 4: Calculate remaining demand after LP optimization
remaining_aisle_lp = scenario1_lp_congestion.set_index("school_id")[
    "aisle_learners_lp"
].to_dict()

lp_redirections_by_flow = redirections_lp.groupby(
    ["school_id_origin", "school_id_destination"]
)["count_redirected_lp"].sum()

flow_remaining = flow_to_congested[
    ["school_id_origin", "school_id_destination", "count_non_beneficiary"]
].copy()
flow_remaining = flow_remaining.merge(
    lp_redirections_by_flow.reset_index(),
    on=["school_id_origin", "school_id_destination"],
    how="left",
)
flow_remaining["count_redirected_lp"] = flow_remaining["count_redirected_lp"].fillna(0)
flow_remaining["remaining_non_benef"] = (
    flow_remaining["count_non_beneficiary"] - flow_remaining["count_redirected_lp"]
).clip(lower=0)
flow_remaining["remaining_aisle"] = (
    flow_remaining["school_id_destination"].map(remaining_aisle_lp).fillna(0)
)
flow_remaining["max_additional"] = flow_remaining[
    ["remaining_non_benef", "remaining_aisle"]
].min(axis=1)

flows_with_demand = flow_remaining[flow_remaining["max_additional"] > 0].copy()
print(f"Flows with remaining redirectable students: {len(flows_with_demand):,}")
print(
    f"Total remaining redirectable students: {flows_with_demand['max_additional'].sum():,.0f}"
)

Flows with remaining redirectable students: 17,020
Total remaining redirectable students: 249,791


In [36]:
# Step 5: Get constrained options and calculate priority scores
# Include size_bonus for consistent priority scoring with main optimization
constrained_options = redirection_options[
    redirection_options["private_esc_id"].isin(constrained_schools)
].copy()
constrained_options = constrained_options[
    [
        "school_id_origin",
        "school_id_destination",
        "private_esc_id",
        "distance_to_private_km",
        "norm_tuition",
        "tuition_jhs",
        "size_bonus",
    ]
].copy()

# Normalize distance within constrained options
dist_min = constrained_options["distance_to_private_km"].min()
dist_max = constrained_options["distance_to_private_km"].max()
constrained_options["norm_distance"] = (
    constrained_options["distance_to_private_km"] - dist_min
) / (dist_max - dist_min + 1e-6)

# Priority score: lower = better (nearer + cheaper + LARGER schools prioritized)
# Formula matches Section 3.7: norm_distance + TUITION_WEIGHT * norm_tuition - SIZE_WEIGHT * size_bonus
constrained_options["priority_score"] = (
    constrained_options["norm_distance"]
    + TUITION_WEIGHT * constrained_options["norm_tuition"]
    - SIZE_WEIGHT * constrained_options["size_bonus"]
)

print(f"Constrained options to process: {len(constrained_options):,}")

Constrained options to process: 284,407


In [37]:
# Step 6: VECTORIZED - Merge flows with constrained options and find best per flow
# Each flow is assigned to exactly ONE best private school option (lowest priority_score)
merged = flows_with_demand[
    ["school_id_origin", "school_id_destination", "max_additional"]
].merge(
    constrained_options[
        [
            "school_id_origin",
            "school_id_destination",
            "private_esc_id",
            "priority_score",
            "tuition_jhs",
        ]
    ],
    on=["school_id_origin", "school_id_destination"],
    how="inner",
)

print(f"Merged flow-options: {len(merged):,}")

if len(merged) > 0:
    # Find best option per flow (minimum priority_score)
    idx_best = merged.groupby(["school_id_origin", "school_id_destination"])[
        "priority_score"
    ].idxmin()
    assignments_df = merged.loc[idx_best].copy()
    assignments_df.rename(
        columns={"max_additional": "unique_blocked_demand"}, inplace=True
    )

    print(f"Flows assigned to constrained schools: {len(assignments_df):,}")
    print(
        f"Total unique blocked demand: {assignments_df['unique_blocked_demand'].sum():,.0f}"
    )

    # Step 7: Aggregate by private school
    school_agg = (
        assignments_df.groupby("private_esc_id")
        .agg(
            {
                "unique_blocked_demand": "sum",
                "school_id_destination": "nunique",
                "school_id_origin": "count",
                "priority_score": "mean",
                "tuition_jhs": "first",
            }
        )
        .reset_index()
    )
    school_agg.columns = [
        "private_esc_id",
        "unique_blocked_demand",
        "num_congested_jhs_affected",
        "num_assigned_flows",
        "avg_priority_score",
        "tuition_jhs",
    ]
else:
    print("No flows could be assigned to constrained schools")
    school_agg = pd.DataFrame(
        columns=[
            "private_esc_id",
            "unique_blocked_demand",
            "num_congested_jhs_affected",
            "num_assigned_flows",
            "avg_priority_score",
            "tuition_jhs",
        ]
    )

Merged flow-options: 194,168
Flows assigned to constrained schools: 13,602
Total unique blocked demand: 219,012


In [39]:
# Step 8: Build final blocked demand analysis table
blocked_demand_analysis = capacity_info.merge(
    school_agg, on="private_esc_id", how="inner"
)

# Add school info
blocked_demand_analysis = blocked_demand_analysis.merge(
    sch_info_slim[["school_id", "school_name", "division", "old_region"]],
    left_on="private_esc_id",
    right_on="school_id",
    how="left",
).drop(columns="school_id")
blocked_demand_analysis.rename(
    columns={"school_name": "private_esc_name", "old_region": "region"}, inplace=True
)

# Add school size data
blocked_demand_analysis = blocked_demand_analysis.merge(
    private_jhs_enrollment[["school_id", "private_jhs_enrollment", "size_bonus"]],
    left_on="private_esc_id",
    right_on="school_id",
    how="left",
).drop(columns="school_id")

# Reorder and sort
blocked_demand_analysis = blocked_demand_analysis[
    [
        "private_esc_id",
        "private_esc_name",
        "division",
        "region",
        "total_slots",
        "available_slots_before",
        "slots_used_lp",
        "utilization_rate",
        "unique_blocked_demand",
        "num_assigned_flows",
        "num_congested_jhs_affected",
        "tuition_jhs",
        "private_jhs_enrollment",
        "size_bonus",
        "avg_priority_score",
    ]
]

blocked_demand_analysis = blocked_demand_analysis.sort_values(
    ["unique_blocked_demand", "avg_priority_score"], ascending=[False, True]
)

# Summary
print("=" * 70)
print("BLOCKED DEMAND ANALYSIS SUMMARY")
print("=" * 70)
print(f"Capacity-constrained schools (>=95% utilized): {len(constrained_schools):,}")
print(f"Schools with blocked demand: {len(blocked_demand_analysis):,}")
print(
    f"Total unique blocked demand: {blocked_demand_analysis['unique_blocked_demand'].sum():,.0f} students"
)
print(f"\nTop 10 schools for slot expansion:")
print("-" * 70)
display(
    blocked_demand_analysis[
        [
            "private_esc_name",
            "division",
            "slots_used_lp",
            "unique_blocked_demand",
            "tuition_jhs",
            "private_jhs_enrollment",
            "size_bonus",
        ]
    ].head(10)
)

BLOCKED DEMAND ANALYSIS SUMMARY
Capacity-constrained schools (>=95% utilized): 931
Schools with blocked demand: 574
Total unique blocked demand: 219,012 students

Top 10 schools for slot expansion:
----------------------------------------------------------------------


,private_esc_name,division,slots_used_lp,unique_blocked_demand,tuition_jhs,private_jhs_enrollment,size_bonus
547,"St. Anthony Learning Institute of Quezon City,...",Quezon City,2.0,3181.0,15375.00,130.0,0.554537
176,Sumulong Memorial High School,Antipolo City,125.0,3001.0,26361.88,1704.0,0.997954
373,"Holy Rosary College Foundation, Inc.",Caloocan City,88.0,2906.0,22024.91,320.0,0.814680
466,Holy Child Catholic School,Manila,111.0,2582.0,45050.00,930.0,0.975979
296,Valley High Academy,Rizal,19.0,2316.0,27750.00,166.0,0.627135
131,Holy Redeemer School,Dasmarinas City,102.0,2198.0,24775.00,261.0,0.758808
467,"Manila Cathedral College, Inc.",Manila,49.0,2148.0,43414.25,1210.0,0.991459
217,St. Gabriel Archangel Academy (Main),Binan City,1.0,1953.0,36300.00,56.0,0.329537
384,Sta. Clara Parish School,Pasay City,49.0,1892.0,46771.22,374.0,0.851157
242,Golden Faith Academy Inc.,Rizal,50.0,1876.0,42300.00,357.0,0.840125


# 7.0. Export Payload for Reports

Export key dataframes for notebook 2.5 (Stakeholder Reports) to consume. This avoids duplicating the optimization workflow.

In [40]:
# Create output directory for scenario1 payload
PAYLOAD_DIR = "output/analysis_payload"
os.makedirs(PAYLOAD_DIR, exist_ok=True)

# 1. Congestion table (public JHS with enrollment, seats, aisle learners)
df_cong.to_parquet(f"{PAYLOAD_DIR}/congestion.parquet", index=False)
print(f"✓ Exported congestion.parquet ({df_cong.shape[0]:,} rows)")

# 2. Greedy redirection log
redirections.to_parquet(f"{PAYLOAD_DIR}/redirections_greedy.parquet", index=False)
print(f"✓ Exported redirections_greedy.parquet ({redirections.shape[0]:,} rows)")

# 3. LP redirection log
redirections_lp.to_parquet(f"{PAYLOAD_DIR}/redirections_lp.parquet", index=False)
print(f"✓ Exported redirections_lp.parquet ({redirections_lp.shape[0]:,} rows)")

# 4. All redirection options (with priority_score, size_bonus, etc.)
redirection_options.to_parquet(
    f"{PAYLOAD_DIR}/redirection_options.parquet", index=False
)
print(f"✓ Exported redirection_options.parquet ({redirection_options.shape[0]:,} rows)")

# 5. ESC slots and availability
esc_available.to_parquet(f"{PAYLOAD_DIR}/esc_available.parquet", index=False)
print(f"✓ Exported esc_available.parquet ({esc_available.shape[0]:,} rows)")

# 6. Filtered student flows (to congested schools)
flow_to_congested.to_parquet(f"{PAYLOAD_DIR}/flow_to_congested.parquet", index=False)
print(f"✓ Exported flow_to_congested.parquet ({flow_to_congested.shape[0]:,} rows)")

# 7. School info (for names/divisions in reports)
sch_info_slim.to_parquet(f"{PAYLOAD_DIR}/school_info.parquet", index=False)
print(f"✓ Exported school_info.parquet ({sch_info_slim.shape[0]:,} rows)")

# 8. Student flow (full, for baseline calculations)
flow.to_parquet(f"{PAYLOAD_DIR}/student_flow.parquet", index=False)
print(f"✓ Exported student_flow.parquet ({flow.shape[0]:,} rows)")

# 9. Blocked demand analysis (slot expansion priorities)
blocked_demand_analysis.to_parquet(
    f"{PAYLOAD_DIR}/blocked_demand_analysis.parquet", index=False
)
print(
    f"✓ Exported blocked_demand_analysis.parquet ({blocked_demand_analysis.shape[0]:,} rows)"
)

# 10. Metadata (parameters and summary stats)
metadata = {
    "max_distance_km": MAX_DISTANCE_KM,
    "tuition_weight": TUITION_WEIGHT,
    "size_weight": SIZE_WEIGHT,
    "cost_penalty": COST_PENALTY,
    "distance_penalty": DISTANCE_PENALTY,
    "size_weight_lp": SIZE_WEIGHT_LP,
    "baseline_aisle_learners": int(baseline_aisle),
    "greedy_aisle_learners": int(greedy_aisle),
    "lp_aisle_learners": int(lp_aisle),
    "greedy_students_redirected": int(redirections["count_redirected"].sum()),
    "lp_students_redirected": int(redirections_lp["count_redirected_lp"].sum()),
    "greedy_schools_used": int(redirections["private_esc_id"].nunique()),
    "lp_schools_used": int(redirections_lp["private_esc_id"].nunique()),
    "constrained_schools": len(constrained_schools),
    "total_blocked_demand": int(blocked_demand_analysis["unique_blocked_demand"].sum()),
}
with open(f"{PAYLOAD_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Exported metadata.json")

print(f"\n{'='*50}")
print(f"Payload exported to: {PAYLOAD_DIR}/")
print(f"{'='*50}")

✓ Exported congestion.parquet (1,098 rows)
✓ Exported redirections_greedy.parquet (5,070 rows)
✓ Exported redirections_lp.parquet (6,162 rows)
✓ Exported redirection_options.parquet (284,670 rows)
✓ Exported esc_available.parquet (1,058 rows)
✓ Exported flow_to_congested.parquet (22,879 rows)
✓ Exported school_info.parquet (61,442 rows)
✓ Exported student_flow.parquet (57,261 rows)
✓ Exported blocked_demand_analysis.parquet (574 rows)
✓ Exported metadata.json

Payload exported to: output/analysis_payload/
